# Генерування атак підміни обличчя (frs-pad) у Google Colab

Блокнот запускає `scripts/generate_attacks.py` на безкоштовному GPU Colab.
Модель – inswapper_128, та сама, що в Deep-Live-Cam.

Порядок роботи:

1. Увімкніть GPU: «Середовище виконання» → «Змінити тип середовища виконання» → T4 GPU.
2. На своєму комп'ютері зберіть архів коду командою `python scripts/pack_colab.py`. Вийде `dist/frs-pad-colab.zip`.
3. Розшифруйте справжні записи учасників командою `python scripts/decrypt_recording.py ...`.
4. Виконайте комірки по черзі. Комірка 2 попросить завантажити архів і відео.

**Захист даних.** Відео учасників – біометричні дані. Завантажуйте їх у Colab
лише тоді, коли учасник у формі згоди погодився на обробку в хмарному сервісі.
Остання комірка видаляє всі відео із середовища.
Моделі InsightFace дозволено використовувати лише для некомерційних досліджень.

In [ ]:
# 1. Залежності (інші пакети в Colab уже є)
!pip -q install onnxruntime-gpu onnx cryptography
import onnxruntime as ort
print(ort.get_available_providers())

In [ ]:
# 2. Завантаження frs-pad-colab.zip і відео
# Потрібні: справжній запис учасника A і фото або запис цільової особи B
from google.colab import files
import zipfile, os
os.makedirs('/content/frs', exist_ok=True)
os.chdir('/content/frs')
up = files.upload()
for name in up:
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('.')
        os.remove(name)
print(sorted(os.listdir('.')))

In [ ]:
# 3. Параметри: заповніть імена файлів
VIDEO_A = 'P02_2026-11-01.webm'        # справжній запис учасника A
SOURCE_B = 'P03_2026-11-02.webm'       # запис (або фото .jpg) цільової особи B
SOURCE_B_ANNOTATION = 'P03_2026-11-02.json'  # анотація запису B; для фото залиште ''
OUT = 'P02_as_P03.mp4'

In [ ]:
# 4. Генерування (моделі ≈ 730 МБ завантажуються з Hugging Face при першому запуску)
src = f'--source-image "{SOURCE_B}"' if SOURCE_B.lower().endswith(('.jpg', '.jpeg', '.png'))       else f'--source-video "{SOURCE_B}"' + (f' --source-annotation "{SOURCE_B_ANNOTATION}"' if SOURCE_B_ANNOTATION else '')
!python scripts/generate_attacks.py "{VIDEO_A}" {src} -o "{OUT}" 

In [ ]:
# 5. Перевірка: перший кадр результату
import cv2
from google.colab.patches import cv2_imshow
cap = cv2.VideoCapture(OUT); cap.set(cv2.CAP_PROP_POS_FRAMES, 60); ok, f = cap.read()
cv2_imshow(cv2.resize(f, None, fx=0.5, fy=0.5))

In [ ]:
# 6. Завантаження результату на свій комп'ютер
files.download(OUT)

In [ ]:
# 7. Видалення відео із середовища Colab
import glob
for p in glob.glob('/content/frs/*'):
    if p.lower().endswith(('.webm', '.mp4', '.jpg', '.jpeg', '.png', '.json')):
        os.remove(p)
print('видалено; також можна «Середовище виконання» → «Від’єднатися та видалити»')